# Ornith Heretic Kaggle API

**Launcher version: 2026-11-04.** Downloads an Ornith-1.5-9B Heretic GGUF and a prebuilt upstream `llama-server` CUDA release, starts an authenticated OpenAI-compatible API, and optionally exposes it through a Cloudflare Quick Tunnel. No source compilation is required.

In [ ]:
# ===== CONFIGURATION | launcher 2026-11-04 =====
MODEL_REPO = "Thunder13240/Ornith-1.5-9B-heretic-GGUF"
MODEL_FILENAME = "Ornith-1.5-9B-heretic-IQ4_XS.gguf"
MODEL_DIR = "/kaggle/working/models"
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp-bin"
CONTEXT_SIZE = 2048
GPU_LAYERS = 999
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
START_CLOUDFLARE_TUNNEL = True

import sys, time, subprocess, secrets, re, os, shutil
from pathlib import Path

## 1. Download the selected model
Checks the live Hugging Face repository file list first, then downloads the best available quant from the configured preference order. No manually attached Kaggle Dataset is required.

In [ ]:
try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi, hf_hub_download

print("Launcher version: 2026-11-04 | dynamic Hugging Face file discovery")
api = HfApi()
repo_files = api.list_repo_files(repo_id=MODEL_REPO, repo_type="model")
gguf_names = [name for name in repo_files if name.lower().endswith(".gguf") and not Path(name).name.lower().startswith("mmproj-")]
if not gguf_names:
    raise FileNotFoundError(f"No text-model GGUF files found in Hugging Face repo: {MODEL_REPO}")

# Match by basename so this still works if the publisher places files in a subfolder.
by_basename = {Path(name).name.lower(): name for name in gguf_names}
requested_basename = Path(MODEL_FILENAME).name.lower()
if requested_basename in by_basename:
    selected_filename = by_basename[requested_basename]
else:
    quant_preferences = [
        "ornith-1.5-9b-heretic-iq4_xs.gguf",
        "ornith-1.5-9b-heretic-q4_k_m.gguf",
        "ornith-1.5-9b-heretic-iq4_nl.gguf",
        "ornith-1.5-9b-heretic-q4_k_s.gguf",
        "ornith-1.5-9b-heretic-iq3_m.gguf",
        "ornith-1.5-9b-heretic-iq3_xs.gguf",
        "ornith-1.5-9b-heretic-iq2_m.gguf",
        "ornith-1.5-9b-heretic-iq2_s.gguf",
    ]
    selected_filename = next((by_basename[name] for name in quant_preferences if name in by_basename), None)
    if selected_filename is None:
        print("Text GGUF files found in the repository:")
        for name in sorted(gguf_names):
            print(" -", name)
        raise FileNotFoundError("No preferred quant was found in the live repository listing; see filenames above.")
    print(f"Configured filename {MODEL_FILENAME!r} was not listed; selected available fallback {selected_filename!r}.")

print("Repository:", MODEL_REPO)
print("Exact file selected from live listing:", selected_filename)
model_path = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=selected_filename,
    local_dir=MODEL_DIR,
))
if not model_path.is_file():
    raise FileNotFoundError(f"Model download did not produce {model_path}")
print("Model ready:", model_path)
print("Model size GiB:", round(model_path.stat().st_size / (2 ** 30), 2))


## 2. Download a prebuilt CUDA `llama-server`
Uses the newest upstream `llama.cpp` release asset built for Ubuntu x64 with CUDA 12.x. That CUDA 12 build is chosen for broader driver compatibility than a CUDA 13.4-only binary; it avoids compiling `llama.cpp` in Kaggle.

In [ ]:
import requests

release_api = "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=100"
release_response = requests.get(release_api, headers={"Accept": "application/vnd.github+json", "User-Agent": "kaggle-llama-server-launcher"}, timeout=30)
release_response.raise_for_status()
releases = release_response.json()

# Prefer the newest release that publishes the Ubuntu x64 CUDA 12 prebuilt bundle.
# CUDA 12.x is intentional: Kaggle's CUDA 13 toolkit does not require using a CUDA 13-built binary.
asset_pattern = re.compile(r"^llama-.*-bin-ubuntu-cuda-12(?:\.\d+)?-x64\.tar\.gz$", re.IGNORECASE)
selected_release = None
selected_asset = None
for release in releases:
    matches = [asset for asset in release.get("assets", []) if asset_pattern.match(asset.get("name", ""))]
    if matches:
        # If multiple CUDA 12 bundles exist in one release, choose the highest minor version.
        def cuda_minor(asset):
            match = re.search(r"cuda-12\.(\d+)-x64", asset["name"], re.IGNORECASE)
            return int(match.group(1)) if match else 0
        selected_release = release
        selected_asset = max(matches, key=cuda_minor)
        break

if selected_asset is None:
    raise RuntimeError("GitHub's recent llama.cpp releases did not expose a matching Ubuntu x64 CUDA 12 bundle. No source build was attempted. Check https://github.com/ggml-org/llama.cpp/releases and rerun.")

print("llama.cpp release:", selected_release.get("tag_name"))
print("Prebuilt asset:", selected_asset["name"])
print("Published:", selected_release.get("published_at"))

llama_dir = Path(LLAMA_CPP_DIR)
if llama_dir.exists():
    shutil.rmtree(llama_dir)
llama_dir.mkdir(parents=True, exist_ok=True)
archive_path = Path("/kaggle/working") / selected_asset["name"]
with requests.get(selected_asset["browser_download_url"], stream=True, timeout=(30, 180)) as download_response:
    download_response.raise_for_status()
    with archive_path.open("wb") as archive_file:
        for chunk in download_response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                archive_file.write(chunk)

print(f"Downloaded bundle: {archive_path.stat().st_size / (1024 * 1024):.1f} MiB")
subprocess.run(["tar", "-xzf", str(archive_path), "-C", str(llama_dir)], check=True)
archive_path.unlink()

server_candidates = list(llama_dir.rglob("llama-server"))
if not server_candidates:
    raise FileNotFoundError(f"The downloaded release archive did not contain llama-server. Extracted files: {[p.name for p in llama_dir.iterdir()][:30]}")
server_binary = server_candidates[0].resolve()
server_binary.chmod(server_binary.stat().st_mode | 0o111)

# Discover all bundled library/backend directories. Recent llama.cpp releases may
# place CUDA backend plugins in a subdirectory rather than beside llama-server.
bundle_lib_dirs = {str(server_binary.parent), str(server_binary.parent / "lib"), str(server_binary.parent / "lib64")}
bundle_lib_dirs.update(str(p.parent) for p in llama_dir.rglob("*.so*") if p.is_file())
bundle_lib_dirs = {p for p in bundle_lib_dirs if Path(p).is_dir()}
existing_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(sorted(bundle_lib_dirs) + ([existing_ld_path] if existing_ld_path else []))
# Tell llama.cpp's dynamic backend loader where the bundled backend libraries live.
os.environ["GGML_BACKEND_PATH"] = ":".join(sorted(bundle_lib_dirs))
print("Backend/library directories:", os.environ["GGML_BACKEND_PATH"])

version_result = subprocess.run([str(server_binary), "--version"], check=True, capture_output=True, text=True, timeout=30, env=os.environ.copy())
print("Verified executable:", server_binary)
print("llama-server version:", (version_result.stdout or version_result.stderr).strip())

## 3. Start the local authenticated API
Starts the downloaded prebuilt `llama-server` with a random API key. If the binary cannot load its libraries, initialize CUDA, or load the model, the cell prints the server log and stops before opening a tunnel.

In [ ]:
try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

# GPU preflight: fail clearly rather than silently serving on CPU.
print("========== KAGGLE GPU CHECK ==========")
nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if nvidia_smi.returncode != 0:
    raise RuntimeError("nvidia-smi failed. In Kaggle, open Notebook settings and select a GPU accelerator, then restart the session and Run All.")
print(nvidia_smi.stdout[:5000])

device_result = subprocess.run([str(server_binary), "--list-devices"], capture_output=True, text=True, timeout=30, env=os.environ.copy())
device_output = (device_result.stdout or "") + "\n" + (device_result.stderr or "")
print("llama.cpp devices:")
print(device_output.strip() or "(no device information returned)")
if device_result.returncode != 0 or not re.search(r"CUDA\s*\d*\s*:", device_output, re.IGNORECASE):
    raise RuntimeError("The downloaded llama-server did not report a CUDA device. Check the Kaggle GPU accelerator and the selected llama.cpp CUDA release bundle; refusing to start an apparently CPU-only API.")

API_KEY = secrets.token_urlsafe(32)
server_log_path = Path("/kaggle/working/llama-server.log")
server_log = server_log_path.open("w")
server_cmd = [str(server_binary), "-m", str(model_path), "--host", SERVER_HOST, "--port", str(SERVER_PORT), "-c", str(CONTEXT_SIZE), "-ngl", str(GPU_LAYERS), "--api-key", API_KEY]
print("Starting llama-server; model loading can take several minutes.")
server_process = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True, env=os.environ.copy())
healthy = False
for _ in range(180):
    if server_process.poll() is not None:
        break
    try:
        response = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=3)
        if response.ok:
            healthy = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)
if not healthy:
    server_log.flush()
    print(server_log_path.read_text(errors="replace")[-12000:])
    raise RuntimeError("llama-server did not become healthy. Cloudflare tunnel was not started.")
print("Local health check passed.")
print("Server log:", server_log_path)
log_text = server_log_path.read_text(errors="replace")
gpu_log_lines = [line for line in log_text.splitlines() if re.search(r"offload|CUDA|GPU|device", line, re.IGNORECASE)]
print("========== GPU OFFLOAD CONFIRMATION ==========")
if gpu_log_lines:
    print("\n".join(gpu_log_lines[-40:]))
else:
    print("No GPU/offload lines found in the server log yet. Inspect the full server log before assuming GPU acceleration is active.")
print("API key created; it is stored in notebook memory and not printed.")


## 4. Test chat completions locally
The API key is sent in the request header and is not printed.

In [ ]:
# Make this test cell safe to run independently after a kernel restart.
try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

test_response = requests.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"model": "local-model", "messages": [{"role": "user", "content": "Reply with exactly: API test passed"}], "max_tokens": 32},
    timeout=300,
)
test_response.raise_for_status()
test_json = test_response.json()
print(test_json.get("choices", [{}])[0].get("message", {}).get("content", "No message content returned"))

## 5. Optionally expose the API through Cloudflare Quick Tunnel
Only run this after the local health check and chat-completion test pass. Quick Tunnel URLs are temporary and change when the tunnel restarts.

In [ ]:
if START_CLOUDFLARE_TUNNEL:
    import platform, urllib.request
    arch = platform.machine().lower()
    if arch not in ("x86_64", "amd64"):
        raise RuntimeError(f"Unsupported architecture for this cloudflared download cell: {arch}")
    cloudflared = Path("/kaggle/working/cloudflared")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
    tunnel_log_path = Path("/kaggle/working/cloudflared.log")
    tunnel_log = tunnel_log_path.open("w")
    tunnel_process = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--url", f"http://{SERVER_HOST}:{SERVER_PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    public_url = None
    for _ in range(60):
        time.sleep(2)
        tunnel_log.flush()
        tunnel_text = tunnel_log_path.read_text(errors="replace")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_text)
        if match:
            public_url = match.group(0)
            break
        if tunnel_process.poll() is not None:
            break
    if not public_url:
        print(tunnel_log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError("Cloudflare Quick Tunnel did not provide a public URL.")
    print("Public API base URL:", public_url)
    print("Use this API key from your client; do not paste it into public logs or commit it to GitHub.")
else:
    print("Tunnel disabled.")

## Client configuration notes
- OpenAI-compatible base URL: append `/v1` to the public tunnel URL.
- API key: the value held in the notebook variable `API_KEY` for this session.
- The notebook must remain running for the endpoint to remain available.
- This is an experimental endpoint. Do not use it for sensitive data. Verify client protocol compatibility and keep the API key private.